# Dynamic RAG, Cleaner Chatbots, and File processing

Last time, we built agent chains with an eye towards a RAG-focused chatbot from web pages.  This time, we're going to look at importing files from other formats to both fill our RAG database and to use as inputs to an LLM chain.

Then we'll pipe the document-based RAG chain back into our very basic chatbot interface so that we can see how the whole thing flows.

Once again, a lot of this work is assembled from the LangChain tutorials, which are really useful.

In [1]:
# Import standard packages
import sys,os, uuid
import json 
import requests, bs4
from tempfile import TemporaryDirectory
from requests.exceptions import RequestException
from pprint import pprint
from parse import parse


# Interactions via JuPyTer
from IPython.display import display, clear_output, Markdown, Image
import ipywidgets as widgets
from ipywidgets import HBox, Label, Layout

# Agent creation wrappers
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from langchain.agents import create_agent

# RAG helpers
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_community.document_loaders import Docx2txtLoader, CSVLoader
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings   # There are others, too.
from langchain_chroma import Chroma
from langchain.agents.middleware import dynamic_prompt, ModelRequest


# Helpers for using tools and contacting humans
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage
from langgraph.types import Command
 
# Additioanl help for understanding and plotting
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

# For CSV handling
import polars as pl

/storage/group/trb21/default/miniforge3/envs/GenAI_Fa2026_Azure/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
my_name = "Timothy_Brick"
endpoint = f"https://genai-fa2026-resource-1.services.ai.azure.com/api/projects/{my_name}_GenAI_Project"
endpoint

'https://genai-fa2026-resource-1.services.ai.azure.com/api/projects/Timothy_Brick_GenAI_Project'

## Set up and test a first agent
As usual, it's worth testing that the environment is set up right.

In [1]:
llm = ChatOpenAI(
    model="Kimi-K2.5",  # Your Azure deployment name
    base_url=endpoint+"/openai/v1",
    api_key=os.environ["PROJECT_API_KEY"],
    temperature=0     # For reproducibility
)
response = llm.invoke("Write a haiku about the challenges of working in Python.")
print(response.content)

NameError: name 'ChatOpenAI' is not defined

# RAG from other documents
The process for importing non-web documents into a RAG vector database is essentially the same as importing from the web, but we use a different import process.  Depending on the exact format you're working with, there are different tools for loading these.  I'm including examples that are common tools here: PDFs and Word .docx files.

When we run these over a set of data, we'll cover CSV files, too--the same lessons apply.

## PDF Read-in

To read in a PDF, we use the same process as with the web:
- Find the appropriate files
- Read them in
- Split them up appropriately
- Create the encoder
- Create the vector DB
- Encode the chunks into the DB

Here, the biggest new piece is reading in the new files.  Luckily, langchain provides loaders for this.  I'm going to briefly show the underlying process, and then we'll show how langchain smooths them over.

### Manual read-in with pypdf

In [4]:
from pypdf import PdfReader
from pathlib import Path

# Path to the folder
pdf_folder= "./GenAIClass-2026/examples/PDFs"

# list all the files
pdf_list = Path(pdf_folder).glob("*.pdf")
all_pdfs = []  # Blank starting point

# Read each file page by page
for pdf_file in pdf_list:
    with PdfReader(pdf_file) as reader:
        this_file = ""
        for page in reader.pages:
            this_file = this_file + page.extract_text()
    all_pdfs.append(this_file)

As always, it helps to spot check and make sure it looks decent.

In [5]:
print(all_pdfs[1][0:400])

Charlie and the Ordovician Ocean
GPT-5.2 Auto
Apr 01, 2026
Charlie loved two things more than anything: science and building machines. So when his
homemade time machine finally hummed to life, he knew exactly where he wanted to go.
“Ready, Comet?” he asked his golden retriever.
Comet barked and jumped into the capsule.
Charlie set the dial to470 million years ago — the Ordovician Period— and press


This is pretty good, although notice the couple of odd tokenization issues.  to470 is rendered as a single word, and "ago - the" has spaces around the mdash, where "Period-" does not.  These kinds of issues are sometimes harmless and are often handled by the encoder, but it's something to look out for if your data aren't coming through well.

For production, careful examination of the inputs is one of the biggest predictors of successful encoding and retrieval, so it's worth working through these carefully.

### Automated read-in with langchain document loaders

As usual, some of this we can hand off to langchain, so instead of doing it the way we did here, we can do it with langchain instead, which will also automatically annotate this with metadata to keep things easy.

The documents here were created for me by ChatGPT-5.2 Auto to be readable by a science-interested middle-schooler.  At a glance, they look simultaneously like they are accurate, like many of the bulleted lists are cribbed almost directly from wikipedia, and like Chat does not think the average middle schooler reads at a particularly high level.

In [8]:
# from langchain_community.document_loaders import PyPDFLoader           # For single files
# from langchain_community.document_loaders import PyPDFDirectoryLoader  # For full folders

# Again, path to the folder
pdf_folder= "./GenAIClass-2026/examples/PDFs"

# The directory loader will pull all pdfs in the folder
document_loader = PyPDFDirectoryLoader(pdf_folder) 

documents = document_loader.load()   # Shockingly, this loads the documents.
print(f"Loaded {len(documents)} documents.")

Loaded 9 documents.


Notice that there are 9 here; that's because each one is a *page*, not a full file.

In [9]:
print(documents[0])

page_content='Charlie and the Forests of the Devonian
GPT-5.2 Auto
Apr 01, 2026
Charlie grinned. “Next stop: 375 million years ago.”
Comet, now wearing tiny protective goggles, wagged his tail.
The time machine shimmered.
The Age of Fishes
They landed beside a wide river. This world looked greener.
The continents were colliding slowly, forming larger landmasses. Warm climates covered
much of the planet.
For the first time in Earth’s history, there wereforests.
Tall early trees likeArchaeopterisstretched into the sky. The ground was covered with ferns
and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish calledplacodermsruled. Some, likeDunkleosteus, were massive
apex predators.
1' metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-04-01T14:17:17-04:00', 'moddate': '2026-04-01T14:17:17-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (Te

Again, notice the errors here. "wereforests" is not a real word (although it's a cool idea).  We can edit this manually in a couple of ways. One would be to go through and manually change them, like this:

In [10]:
documents[0].page_content = documents[0].page_content.replace("wereforests", "were forests")
print(documents[0].page_content)

Charlie and the Forests of the Devonian
GPT-5.2 Auto
Apr 01, 2026
Charlie grinned. “Next stop: 375 million years ago.”
Comet, now wearing tiny protective goggles, wagged his tail.
The time machine shimmered.
The Age of Fishes
They landed beside a wide river. This world looked greener.
The continents were colliding slowly, forming larger landmasses. Warm climates covered
much of the planet.
For the first time in Earth’s history, there were forests.
Tall early trees likeArchaeopterisstretched into the sky. The ground was covered with ferns
and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish calledplacodermsruled. Some, likeDunkleosteus, were massive
apex predators.
1


But this is an AI class, so we'll have Kimi do it for us.

In [11]:
# Prompt structure:
FIX_TOKENIZATION_PROMPT = ("You are a helpful AI bot that fixes tokenization errors"
                           " that result from scraping text out of PDF files. Review"
                           " the text below and identify all the places where two words"
                           " were incorrectly combined into a single word.  Replace all of"
                           " the combined words with the correctly split words. Return"
                           " the exact text that the user submitted, but with the corrected"
                           " spacing. Do not return any other answer."
                           " The text is:"
                           " {text}"
                          )
# Test:
response = llm.invoke(FIX_TOKENIZATION_PROMPT.format(text="This is incorrectlyformattedtext that should beseparated."))
print(response.content)

This is incorrectly formatted text that should be separated.


Now we can loop over all of our documents and fix them.  Notice the *for..in* loop here.  
We've used these before offhandedly, but not deeply. As a reminder for those new to programming, the indented part inside the loop will run once for each document in refined_docs, calling the current document a_doc each time.

In [12]:
refined_docs = documents
for a_doc in refined_docs:
    print(f"Correcting {a_doc.metadata["source"]}, page {a_doc.metadata["page_label"]}....")
    a_doc.page_content = llm.invoke(FIX_TOKENIZATION_PROMPT.format(text=a_doc.page_content)).content
    print("...Done.")

Correcting GenAIClass-2026/examples/PDFs/Devonian.pdf, page 1....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Devonian.pdf, page 2....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Devonian.pdf, page 3....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Ordovician.pdf, page 1....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Ordovician.pdf, page 2....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Ordovician.pdf, page 3....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Permian.pdf, page 1....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Permian.pdf, page 2....
...Done.
Correcting GenAIClass-2026/examples/PDFs/Permian.pdf, page 3....
...Done.


In [13]:
print(refined_docs[0])

page_content='Charlie and the Forests of the Devonian
GPT-5.2 Auto
Apr 01, 2026
Charlie grinned. “Next stop: 375 million years ago.”
Comet, now wearing tiny protective goggles, wagged his tail.
The time machine shimmered.
The Age of Fishes
They landed beside a wide river. This world looked greener.
The continents were colliding slowly, forming larger landmasses. Warm climates covered
much of the planet.
For the first time in Earth’s history, there were forests.
Tall early trees like Archaeopteris stretched into the sky. The ground was covered with ferns
and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish called placoderms ruled. Some, like Dunkleosteus, were massive
apex predators.
1' metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-04-01T14:17:17-04:00', 'moddate': '2026-04-01T14:17:17-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.

Notice also that the metadata here isn't the best.  It added everything it had right away, but not actually the kind of useful information we might want in practice.  We can modify the metadata with a similar loop.  For example, a better layout might be to pull out just the title (on the first line) and the era that the document is talking about.  We can either do that by adding that info to the loader, or we can do it separately after.

This is more complex code; you can either read through it to understand it in detail, or think of it as a tool that you can use later.

In [22]:
# create a dictionary (named list) of pages by source for easy lookup:
pages_by_source = {}
annotations_by_source = {}
for a_page in refined_docs:
    source = a_page.metadata["source"]
    if source not in pages_by_source.keys():  # New source!
        # Get the period based on the file name using python Parse:        
        period = parse("GenAIClass-2026/examples/PDFs/{}.pdf", source)[0]
        pages_by_source[source] = []                       # Create empty entry for pages
        annotations_by_source[source] = {"Period":period}  # Create entry for annotations with content
    pages_by_source[source].append(a_page)                   # Add the page to the pages list
    
    # if this is the first page, get the title from the first line
    if a_page.metadata["page_label"] == "1":
        annotations_by_source[source]["title"] = a_page.page_content.splitlines()[0]

In [21]:
source = a_page.metadata["source"]
print(source)

GenAIClass-2026/examples/PDFs/Devonian.pdf


In [23]:
# Now add annotations and appropriately sort the pages
sourced_pages = []
for source, pages in pages_by_source.items():
    for this_page in pages:
        this_page.metadata.update(annotations_by_source[source])
        sourced_pages.append(this_page)

Now we're set, and we can split up our documents using our classic RecursiveCharacterTextSplitter.

In [24]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # chunk size (characters)
    chunk_overlap=100,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(sourced_pages)

In [25]:
print(all_splits[0])

page_content='Charlie and the Forests of the Devonian
GPT-5.2 Auto
Apr 01, 2026
Charlie grinned. “Next stop: 375 million years ago.”
Comet, now wearing tiny protective goggles, wagged his tail.' metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-04-01T14:17:17-04:00', 'moddate': '2026-04-01T14:17:17-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'source': 'GenAIClass-2026/examples/PDFs/Devonian.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'Period': 'Devonian', 'title': 'Charlie and the Forests of the Devonian', 'start_index': 0}


Notice that all of the metadata from the PDF is included in the output; this will come in handy for the retriever when looking for and referencing the files again.

### Vector Embeddings
Nothing new here; we need an embedding model to be our process model.

In [26]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Your Azure deployment name
    base_url="https://GenAI-Fa2026-Resource-1.cognitiveservices.azure.com/openai/v1",
    api_key=os.environ["PROJECT_API_KEY"]
)

We're still using databases in /tmp/, but you can change this any time by changing the RAG_ROOT.

In [27]:
# Where does it go?
# RAG_ROOT = os.environ["RAG_DATAROOT"]
# RAG_ROOT = "~/work/"
tempdir = TemporaryDirectory(ignore_cleanup_errors=True)    # Delete this line for one that will stay somewhere
RAG_ROOT = tempdir.name                                     # This one, too.
RAG_ROOT = RAG_ROOT + "/newProject.db"  # Replace the "newProject" with a project name, if you want.
print(RAG_ROOT)

/tmp/tmp3ay03fxi/newProject.db


Set up the database there:

In [28]:
rag_vectors = Chroma(
    collection_name="Charlies_Travels",  # Note: spaces and many punctuation marks are not allowed here
    embedding_function=embeddings,
    persist_directory=RAG_ROOT,  # Where to save data locally, remove if not necessary
)

In [29]:
m = rag_vectors.add_documents(all_splits) # Stored as m to avoid printing to screen.

This will work the same as the RAG databases we used previously, so we can build up our RAG retriever tool the same way as we did last time, only this time we'll augment the 

In [30]:
rag_vectors = Chroma(
    collection_name="Charlies_Travels",  # Note: spaces and many punctuation marks are not allowed here
    embedding_function=embeddings,
    persist_directory=RAG_ROOT,  # Where to save data locally, remove if not necessary
)

m = rag_vectors.add_documents(all_splits) # Stored as m to avoid printing to screen.

retriever = rag_vectors.as_retriever()

def get_charlie_info(query) :
    docs=retriever.invoke(query)
    docs_info = []
    for doc in docs:
        # Note: Try out different formats for this!
        docs_info.append(
                   f"title:{doc.metadata["title"]}\n"
                   f"Period:{doc.metadata['Period']}\n"
                   f"content:{doc.page_content}"
                   )
    return "\n\n".join(docs_info)
# Make a tool:
@tool
def charlie_info_tool(
    query:str
) -> str:
    """ Get information about Penn State policies."""
    return get_charlie_info(query)
    
print(charlie_info_tool.invoke({"query":"animals"}))

title:Charlie and the Great Dying
Period:Permian
content:• About 70% of land vertebrate species extinct.
“This is the worst one,” Charlie said quietly.
Comet pressed close.
“But life will recover,” Charlie said. “And eventually, dinosaurs will rise.”

title:Charlie and the Great Dying
Period:Permian
content:• About 70% of land vertebrate species extinct.
“This is the worst one,” Charlie said quietly.
Comet pressed close.
“But life will recover,” Charlie said. “And eventually, dinosaurs will rise.”

title:Charlie and the Forests of the Devonian
Period:Devonian
content:and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish called placoderms ruled. Some, like Dunkleosteus, were massive

title:Charlie and the Forests of the Devonian
Period:Devonian
content:and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish called placoderms ruled. Some, lik

In [31]:
# Make a tool:
@tool
def charlie_info_tool(
    query:str
) -> str:
    """ Get information about Penn State policies."""
    return get_charlie_info(query)
    
print(charlie_info_tool.invoke({"query":"animals"}))

title:Charlie and the Great Dying
Period:Permian
content:• About 70% of land vertebrate species extinct.
“This is the worst one,” Charlie said quietly.
Comet pressed close.
“But life will recover,” Charlie said. “And eventually, dinosaurs will rise.”

title:Charlie and the Great Dying
Period:Permian
content:• About 70% of land vertebrate species extinct.
“This is the worst one,” Charlie said quietly.
Comet pressed close.
“But life will recover,” Charlie said. “And eventually, dinosaurs will rise.”

title:Charlie and the Forests of the Devonian
Period:Devonian
content:and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish called placoderms ruled. Some, like Dunkleosteus, were massive

title:Charlie and the Forests of the Devonian
Period:Devonian
content:and primitive plants.
“There’s soil!” Charlie said excitedly. “Real ecosystems on land!”
Life Expands
In the water, armored fish called placoderms ruled. Some, lik

This retriever can be embedded wherever we want it.  Here's a simple agent that uses the retriever directly as a tool.

In [55]:
CHARLIE_BOT_PROMPT = ("You answer questions about some short stories."
                      " Use the get_charlie_info tool to get info about"
                      " what happens in those stories, and respond sensibly."
                     )
rag_agent = create_agent(llm, system_prompt=CHARLIE_BOT_PROMPT, tools=[charlie_info_tool])

In [33]:
query = "Which book has Charlie encounter Dimetrodon?"
response = rag_agent.invoke({"messages": [{"role": "user", "content": query}]})
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Charlie encounters Dimetrodon in **"Charlie and the Great Dying"**. 

In the story, which is set during the Permian period, Charlie spots Dimetrodon in the distance and reminds Comet that it's not a dinosaur, but rather a synapsid (mammal-like reptile) that lived before dinosaurs.


But we could incorporate this into a more complex model, too.

# CSV: The Data 

CSV files are loaded in a manner similar to R.  You can load them with a langchain CSVLoader if you want to, in which case they are serialized into JSON--that is, each row is handled as a list of column_name:column_value pairs and encoded that way.

Here, though, we want to have the csv file as a data frame, which will be better for looping over the data set.  In python, the new data frame hotness is a package called *polars*, which has a feel not too different from R data frames.  You can read about them [here](https://kevinheavey.github.io/modern-polars/) if you want to.  I'm still getting used to these, so I asked Claude for a some help figuring out how to do this cleanly.

Note that pandas' read_csv function makes you specify if there's a quotation character to quote strings that might have commas in them--this can be annoying.

The CSV file we're using here has a list of animals that we'll use for our classification example.  

In [35]:
all_animals = pl.read_csv("GenAIClass-2026/examples/PDFs/Animal_Eras_Tour.csv", quote_char="'")
animals = all_animals.head(5)
animals

Period,Taxonomic_Class,Name,Diet,Apex,Environment,Kingdom_Category,Encountered_by_Charlie
str,str,str,str,bool,str,str,bool
"""Ordovician""","""Arthropod""","""Trilobite""","""Herbivore""",false,"""Marine""","""Animal""",true
"""Ordovician""","""Brachiopod""","""Brachiopod""","""Herbivore""",false,"""Marine""","""Animal""",true
"""Ordovician""","""Echinoderm""","""Crinoid""","""Herbivore""",false,"""Marine""","""Animal""",true
"""Ordovician""","""Cnidarian""","""Early coral""","""Carnivore""",false,"""Marine""","""Animal""",true
"""Ordovician""","""Mollusk""","""Nautiloid cephalopod""","""Carnivore""",true,"""Marine""","""Animal""",true


Given the slightly young target we have here, we'll first take a zero-shot approach to classifying whether they are carnivores or herbivores.  Note that the "ground truth" I'm using here is AI-generated, so I expect Kimi to do fairly well here.

First, we'll want to pull out the list of animal names and their diets, so we can see them.  The commands here looks a lot like tidyverse, if you're accustomed to that in R, but with a . instead of a |>.  Many of the tidyverse verbs (like .select()) are available here. Note that []s are still used for manual indexing.

In [36]:
animals["Name"]

Name
str
"""Trilobite"""
"""Brachiopod"""
"""Crinoid"""
"""Early coral"""
"""Nautiloid cephalopod"""


Danger warning: Python is 0-indexed, so indexes start at the 0th row, not the 1st.

In [37]:
display(animals[1])  # Not the first row!
display(animals[0])  # This one is!

Period,Taxonomic_Class,Name,Diet,Apex,Environment,Kingdom_Category,Encountered_by_Charlie
str,str,str,str,bool,str,str,bool
"""Ordovician""","""Brachiopod""","""Brachiopod""","""Herbivore""",false,"""Marine""","""Animal""",true


Period,Taxonomic_Class,Name,Diet,Apex,Environment,Kingdom_Category,Encountered_by_Charlie
str,str,str,str,bool,str,str,bool
"""Ordovician""","""Arthropod""","""Trilobite""","""Herbivore""",false,"""Marine""","""Animal""",true


In any case where we need to walk through the whole set, we do it with a loop:

In [38]:
for name in animals["Name"]:
    print(name)

Trilobite
Brachiopod
Crinoid
Early coral
Nautiloid cephalopod


Following the examples from zero shot (from many weeks ago), we'll create an agent with a zero-shot classification prompt first.

In [39]:
ZERO_SHOT_ANIMAL_PROMPT = ("Classify the animal listed below as either a"
                           " carnivore or an herbivore. Respond with only the"
                           " word Herbivore or Carnivore (with the first letter"
                           " capitalized). Do not reply with any other word."
                           " The animal is:"
                           " {entry} "
                          )
# Let's test it once:
response=llm.invoke(ZERO_SHOT_ANIMAL_PROMPT.format(entry="Dog"))
response.content

'Carnivore'

That looks pretty good. So let's take a look at how it does across the whole data set.  Normally we'd separate out a small chunk to make sure we're doing things right, but let's give it a try for now.

In [40]:
guesses = []    # Zero out to start

# Walk through animals, making and storing a guess each time.
for animal in animals["Name"]:
    print(f"Guessing about: {animal}.")
    response = llm.invoke(ZERO_SHOT_ANIMAL_PROMPT.format(entry=animal))
    guess = response.content
    guesses.append(guess)

# Appending columns is a pain, but it looks like this:
animals_with_guesses = animals.with_columns(pl.Series("Guess", guesses))

Guessing about: Trilobite.
Guessing about: Brachiopod.
Guessing about: Crinoid.
Guessing about: Early coral.
Guessing about: Nautiloid cephalopod.


In [41]:
animals_with_guesses.select("Name", "Diet", "Guess")

Name,Diet,Guess
str,str,str
"""Trilobite""","""Herbivore""","""Carnivore"""
"""Brachiopod""","""Herbivore""","""Herbivore"""
"""Crinoid""","""Herbivore""","""Carnivore"""
"""Early coral""","""Carnivore""","""Carnivore"""
"""Nautiloid cephalopod""","""Carnivore""","""Carnivore"""


## Adding RAG for classification

As we did last week, we can make a dynamic prompting agent that uses the RAG retriever to improve its prompts.

In [83]:
@dynamic_prompt
def prompt_with_charlie_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    charlie_info = get_charlie_info(query)

    system_message = ("Classify the animal or plant in the user input based on"
                      " the period of Earth's history in which it appeared."
                      " Use the get_charlie_info tool to discover which"
                      " time periods Charlie encountered or discussed"
                      " each animal to help you with your answer."
                      " Respond with the Period of history,"
                      " capitalized (e.g. Devonian )."
                      "Entry:\n"
                      f" {last_query}\n\n"
                      ""
                      "Context:\n"
                      f"{charlie_info}"
                     )
    print("using charlie tool")
    return system_message

In [72]:
animal_rag_agent = create_agent(llm, tools=[], middleware=[prompt_with_charlie_context])

In [73]:
response = animal_rag_agent.invoke({"messages":[{"role":"user", "content":"Trilobites"}]})
response["messages"][-1].content

'Ordovician'

## Do it yourself: use this bot to classify animals

As before, we can assemble these elements together.  Following our chain example from Tuesday, let's build a chain that classifies time based on the Charlie stories.

In [50]:
rag_system_instructions = ("Classify the animal or plant in the user input based on"
                      " the period of Earth's history in which it appeared."
                      " Use the prompt_with_charlie_context tool to discover which"
                      " time periods Charlie encountered or discussed"
                      " each animal to help you with your answer."
                      " Respond ONLY with the Period of history,"
                      " capitalized (e.g. Devonian )."
                      " The animal is:"
                      " {entry} ")

In [46]:
animals["Name"]

Name
str
"""Trilobite"""
"""Brachiopod"""
"""Crinoid"""
"""Early coral"""
"""Nautiloid cephalopod"""


In [ ]:
guesses = []    # Zero out to start

# Walk through animals, making and storing a guess each time.
for animal in animals["Name"]:
    print(f"Guessing about: {animal}.")
    response = animal_rag_agent.invoke({"messages":[{"role":"user", "content":f"{animal}"}]})
    print(response)
    guess = response["messages"][-1].content
    guesses.append(guess)

# Appending columns is a pain, but it looks like this:
animals_with_guesses = animals.with_columns(pl.Series("Guess", guesses))

Guessing about: Trilobite.
{'messages': [HumanMessage(content='Trilobite', additional_kwargs={}, response_metadata={}, id='e2154d04-8fbf-40be-ab0c-16fe81ed7348'), AIMessage(content='Cambrian', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 841, 'prompt_tokens': 346, 'total_tokens': 1187, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'audio_prompt_tokens': 0, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'Kimi-K2.5', 'system_fingerprint': None, 'id': '032e9578d665491fb71eff2efb7a7ef6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d68f4-649b-7903-b7b5-8318284e555e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 346, 'output_tokens': 841, 'total_tokens': 1187, 'input_token_details': {}, 'output_token_details': {}})]}
Guessing about: Brachiopod.
{'messages': [HumanMessage(content='Brachiopod', additional_kwargs={}, response_metadata={}, id='53e2fba0-08bb-4663-938f-b

In [88]:
print(response)

{'messages': [HumanMessage(content='Nautiloid cephalopod', additional_kwargs={}, response_metadata={}, id='8f0b9d68-3684-429a-b9dd-7fb681ccf91c'), AIMessage(content='Permian', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1158, 'prompt_tokens': 354, 'total_tokens': 1512, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'audio_prompt_tokens': 0, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'Kimi-K2.5', 'system_fingerprint': None, 'id': 'fbc3ec7bc6ee4e969481931eea2b2ef6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d68f4-18e7-7633-9e17-26a70af46811-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 354, 'output_tokens': 1158, 'total_tokens': 1512, 'input_token_details': {}, 'output_token_details': {}})]}


In [85]:
animals_with_guesses.select("Name", "Period", "Guess")

Name,Period,Guess
str,str,str
"""Trilobite""","""Ordovician""","""Cambrian"""
"""Brachiopod""","""Ordovician""","""Cambrian"""
"""Crinoid""","""Ordovician""","""Ordovician"""
"""Early coral""","""Ordovician""","""Cambrian"""
"""Nautiloid cephalopod""","""Ordovician""","""Ordovician"""
